In [7]:
import sys
import requests
import pandas as pd
import numpy as np
import time

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("requests", requests.__version__)
print("pandas", pd.__version__)

Python: /Users/facu/Desktop/UCC.nosync/2026/3. SecondSemester/Estadistica Multivariada/4.Proyecto Ingredor.nosync/venv/.venv/bin/python
NumPy: 2.0.2
requests 2.32.5
pandas 2.3.3


In [2]:
#coneccion con la base address.
BASE = "https://api.motogp.pulselive.com/motogp/v1"

#chequeo endpoint seasons. 
url_seasons = BASE + "/results/seasons"
respuesta_seasons = requests.get(url_seasons, timeout=30)

print("Código de estado:", respuesta_seasons.status_code)

Código de estado: 200


In [3]:
# Ver que trajo la request a url_seasons.

datos = respuesta_seasons.json()

print("Tipo de dato:", type(datos))
print("Cantidad de elementos (temporadas totales registradas):", len(datos))
print("Primer elemento:", datos[0])

Tipo de dato: <class 'list'>
Cantidad de elementos (temporadas totales registradas): 78
Primer elemento: {'id': 'e88b4e43-2209-47aa-8e83-0e0b1cedde6e', 'name': None, 'year': 2026, 'current': True}


In [4]:
# Convertir doc json (lista) en una tabla con shape

df_seasons = pd.DataFrame(datos)

print(df_seasons.shape)
df_seasons.head()

(78, 4)


,id,name,year,current
0,e88b4e43-2209-47aa-8e83-0e0b1cedde6e,None,2026,True
1,ae6c6f0d-c652-44f8-94aa-420fc5b3dab4,None,2025,False
2,dd12382e-1d9f-46ee-a5f7-c5104db28e43,None,2024,False
3,db8dc197-c7b2-4c1b-b3a4-6dc534c023ef,None,2023,False
4,db8dc197-c7b2-4c1b-b3a4-6dc534c014ef,None,2022,False


In [5]:
# comprobar valores null. 

df_seasons.isna().sum()

id          0
name       78
year        0
current     0
dtype: int64

In [6]:
# filtro temporadas [2015, 2025]

df_seasons = df_seasons[(df_seasons["year"] >= 2015) & (df_seasons["year"] <= 2025)]
df_seasons = df_seasons.sort_values("year").reset_index(drop=True)

print(df_seasons.shape)
df_seasons[["year", "id", "current"]]

(11, 4)


,year,id,current
0,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,False
1,2016,ba354746-c4da-4dc8-8b8e-776fcc43287d,False
2,2017,e24e8a8c-4673-498d-86e0-731581d7c586,False
3,2018,c0844c71-783c-4f86-abce-a9a67adcc1a9,False
4,2019,cbc5dc8d-3aae-4327-bd88-d1cdeea9b654,False
5,2020,fa729c5c-a34a-40b9-8ebf-2578d9433517,False
6,2021,db8dc197-c7b2-4c1b-b3a4-7dc723c087ed,False
7,2022,db8dc197-c7b2-4c1b-b3a4-6dc534c014ef,False
8,2023,db8dc197-c7b2-4c1b-b3a4-6dc534c023ef,False
9,2024,dd12382e-1d9f-46ee-a5f7-c5104db28e43,False


In [8]:
# pedir funcion que consulta a la base de datos segun endpoint. return documentos json.

def pedir(ruta, params=None):
    r = requests.get(BASE + ruta, params=params, timeout=30)
    time.sleep(0.3)
    if r.status_code != 200:
        print("Error", r.status_code, r.url)
        return None
    return r.json()

In [9]:
# prueba categorias 
season_id = df_seasons.loc[0, "id"]
cats = pedir("/results/categories", {"seasonUuid": season_id})
pd.DataFrame(cats)

,id,name,legacy_id
0,e8c110ad-64aa-4e8e-8a86-f2f152f6a942,MotoGP™,3
1,549640b8-fd9c-4245-acfd-60e4bc38b25c,Moto2™,2
2,954f7e65-2ef2-4423-b949-4961cc603e45,Moto3™,1


In [10]:
# recorro cada categoria. Imprimo total de temporadas
cat_motogp = {}

for _, fila in df_seasons.iterrows():
    cats = pedir("/results/categories", {"seasonUuid": fila["id"]})
    for c in cats or []:
        if c["legacy_id"] == 3:
            cat_motogp[fila["id"]] = c["id"]

print("Temporadas con MotoGP:", len(cat_motogp))

Temporadas con MotoGP: 11


In [11]:
evs = pedir("/results/events", {"seasonUuid": season_id, "isFinished": "true"})

print("Cantidad de eventos:", len(evs))
print("Campos disponibles:", list(evs[0].keys()))

Cantidad de eventos: 27
Campos disponibles: ['country', 'event_files', 'circuit', 'test', 'sponsored_name', 'date_end', 'toad_api_uuid', 'date_start', 'additional_name', 'name', 'legacy_id', 'season', 'short_name', 'id', 'status']


In [15]:
#armo tabla de eventos de las 11 temporadas
eventos = []

for _, fila in df_seasons.iterrows():
    evs = pedir("/results/events", {"seasonUuid": fila["id"], "isFinished": "true"})
    for e in evs or []:
        eventos.append({
            "year": fila["year"],
            "season_id": fila["id"],
            "event_id": e.get("id"),
            "event_name": e.get("name"),
            "short_name": e.get("short_name"),
            "test": e.get("test"),
            "country": (e.get("country") or {}).get("name"),
            "circuit": (e.get("circuit") or {}).get("name"),
            "date_start": e.get("date_start"),
            "date_end": e.get("date_end"),
            "status": e.get("status"),
        })

df_events = pd.DataFrame(eventos)
print(df_events.shape)
df_events.head()

(309, 11)


,year,season_id,event_id,event_name,short_name,test,country,circuit,date_start,date_end,status
0,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,c68ece2b-fc82-42b1-a1fb-6a0f4af01d64,GRAND PRIX OF QATAR,QAT,False,Qatar,Lusail International Circuit,2015-03-26,2015-03-29,FINISHED
1,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,902eaad6-a45f-4afd-8ea2-9a409ecc6984,GRAND PRIX OF THE AMERICAS,AME,False,United States of America,Circuit Of The Americas,2015-04-10,2015-04-12,FINISHED
2,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,a4b6173e-c2b7-4a26-b336-91e025aecfbe,GRAN PREMIO DE LA REPÚBLICA ARGENTINA,ARG,False,Argentina,Termas de Río Hondo,2015-04-17,2015-04-19,FINISHED
3,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,4862fda0-e20d-401a-b844-51a1987a7358,GRAN PREMIO DE ESPAÑA,SPA,False,Spain,Circuito de Jerez - Ángel Nieto,2015-05-01,2015-05-03,FINISHED
4,2015,cbe871d8-d8fa-47cb-8b63-cabe0015223e,344ffc63-16b3-4660-8e51-14332d308fd1,GRAND PRIX DE FRANCE,FRA,False,France,Le Mans,2015-05-15,2015-05-17,FINISHED


In [16]:
#separa carreras de test. True = test ; False = fin de semana de carrera 

df_events["test"].value_counts(dropna=False)

test
False    207
True     102
Name: count, dtype: int64

In [17]:
# me quedo solo con las carreras 

df_events = df_events[df_events["test"] == False].reset_index(drop=True)

print(df_events.shape)
df_events.groupby("year").size()

(207, 11)


year
2015    18
2016    18
2017    18
2018    19
2019    19
2020    15
2021    18
2022    20
2023    20
2024    20
2025    22
dtype: int64

In [18]:
#Paso 14: sesiones de un evento (prueba). Pedimos sesiones de ese evento

ev = df_events.iloc[0]
cat_id = cat_motogp[ev["season_id"]]

ses = pedir("/results/sessions", {"eventUuid": ev["event_id"], "categoryUuid": cat_id})

df_ses_test = pd.DataFrame(ses)
print(ev["year"], ev["event_name"])
df_ses_test[["id", "type", "number", "date", "status"]]

2015 GRAND PRIX OF QATAR


,id,type,number,date,status
0,a3fecda7-6217-4b16-8fbe-85cc7756cdea,FP,1.0,2015-03-26T19:55:00+00:00,FINISHED
1,c17fc8bc-2bf2-44cf-b984-fdaeb50c3d41,FP,2.0,2015-03-27T18:00:00+00:00,FINISHED
2,9e949114-158f-4fba-80ab-11691f2b888e,FP,3.0,2015-03-27T20:55:00+00:00,FINISHED
3,f1845a9d-c4ec-45e1-b422-23e164f2aa55,FP,4.0,2015-03-28T19:55:00+00:00,FINISHED
4,5528c9a7-7c86-4272-bbc3-84bdf0618078,Q,1.0,2015-03-28T20:35:00+00:00,FINISHED
5,fa0dfa2d-0127-451e-b031-bc232b88fcbf,Q,2.0,2015-03-28T21:00:00+00:00,FINISHED
6,518a646a-5418-4f8d-91f7-6c569be5b3a0,WUP,NaN,2015-03-29T17:00:00+00:00,FINISHED
7,b813e5b1-d4e9-4bd1-a18f-a722e27633e2,RAC,NaN,2015-03-29T21:00:00+00:00,FINISHED


In [19]:
#Paso 15: ver la velocidad en una sesión

sesion_id = df_ses_test.loc[0, "id"]   # <- cambiá el 0 por la fila de la carrera

d = pedir(f"/results/session/{sesion_id}/classification",
          {"seasonYear": ev["year"], "test": "false"})

filas = []
for r in d["classification"]:
    filas.append({
        "piloto": r["rider"]["full_name"],
        "posicion": r["position"],
        "moto": r["constructor"]["name"],
        "top_speed": r["top_speed"],
        "vueltas": r["total_laps"],
        "status": r["status"],
    })

df_clas_test = pd.DataFrame(filas)
df_clas_test

,piloto,posicion,moto,top_speed,vueltas,status
0,Marc Marquez,1,Honda,343.7,15,INSTND
1,Daniel Pedrosa,2,Honda,347.3,15,INSTND
2,Aleix Espargaro,3,Suzuki,328.8,13,INSTND
3,Jorge Lorenzo,4,Yamaha,338.4,17,INSTND
4,Cal Crutchlow,5,Honda,340.2,16,INSTND
5,Danilo Petrucci,6,Ducati,341.2,15,INSTND
6,Andrea Dovizioso,7,Ducati,345.0,17,INSTND
7,Yonny Hernandez,8,Ducati,343.5,14,INSTND
8,Valentino Rossi,9,Yamaha,338.4,18,INSTND
9,Andrea Iannone,10,Ducati,344.4,17,INSTND


In [21]:
#Paso 16: revisar qué tan completa viene la velocidad

df_clas_test["top_speed"].describe()
df_clas_test["top_speed"].isna().sum()

np.int64(0)

In [22]:
#Celda 1: pedir con memoria en disco. Son mas de 2000 pedidos.

import json, hashlib
from pathlib import Path

RAW = Path("..") / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

def pedir(ruta, params=None):
    clave = hashlib.md5((ruta + str(sorted((params or {}).items()))).encode()).hexdigest()
    archivo = RAW / f"{clave}.json"
    if archivo.exists():
        return json.loads(archivo.read_text(encoding="utf-8"))
    r = requests.get(BASE + ruta, params=params, timeout=30)
    time.sleep(0.3)
    if r.status_code != 200:
        print("Error", r.status_code, r.url)
        return None
    data = r.json()
    archivo.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    return data

In [25]:
#Celda 2: todas las sesiones de MotoGP

sesiones = []
for _, ev in df_events.iterrows():
    cat_id = cat_motogp[ev["season_id"]]
    ss = pedir("/results/sessions", {"eventUuid": ev["event_id"], "categoryUuid": cat_id})
    for s in ss or []:
        cond = s.get("condition") or {}
        sesiones.append({
            "year": ev["year"], "event_id": ev["event_id"], "circuito": ev["short_name"],
            "session_id": s.get("id"), "type": s.get("type"), "number": s.get("number"),
            "date": s.get("date"), "status": s.get("status"),
            "pista": cond.get("track"), "aire": cond.get("air"), "humedad": cond.get("humidity"),
            "suelo": cond.get("ground"), "clima": cond.get("weather"),
        })

df_ses = pd.DataFrame(sesiones)
print(df_ses.shape)
print(df_ses["type"].value_counts())

(1651, 13)
type
FP     690
Q      412
RAC    210
WUP    206
SPR     62
PR      54
P       16
TTS      1
Name: count, dtype: int64


In [27]:
#Celda 3: clasificación de todas las sesiones, 2015-2025

filas = []
for i, (_, s) in enumerate(df_ses.iterrows()):
    d = pedir(f"/results/session/{s['session_id']}/classification",
              {"seasonYear": s["year"], "test": "false"})
    if not isinstance(d, dict):
        continue
    for r in d.get("classification") or []:
        rider = r.get("rider") or {}
        team = r.get("team") or {}
        cons = r.get("constructor") or {}
        bl = r.get("best_lap") or {}
        gap = r.get("gap") or {}
        filas.append({
            "year": s["year"], "event_id": s["event_id"], "circuito": s["circuito"],
            "session_id": s["session_id"], "type": s["type"], "number": s["number"],
            "rider_id": rider.get("id"), "rider_legacy_id": rider.get("legacy_id"),
            "piloto": rider.get("full_name"), "pais": (rider.get("country") or {}).get("name"),
            "equipo": team.get("name"), "fabricante": cons.get("name"),
            "posicion": r.get("position"), "mejor_vuelta": bl.get("time"),
            "vuelta_mejor": bl.get("number"), "vueltas": r.get("total_laps"),
            "top_speed": r.get("top_speed"), "gap_primero": gap.get("first"),
            "gap_previo": gap.get("prev"), "status": r.get("status"),
        })
    if (i + 1) % 100 == 0:
        print(i + 1, "de", len(df_ses), "sesiones")

df_clas = pd.DataFrame(filas)
print(df_clas.shape)

100 de 1651 sesiones
200 de 1651 sesiones
300 de 1651 sesiones
400 de 1651 sesiones
500 de 1651 sesiones
600 de 1651 sesiones
700 de 1651 sesiones
800 de 1651 sesiones
900 de 1651 sesiones
1000 de 1651 sesiones
1100 de 1651 sesiones
1200 de 1651 sesiones
1300 de 1651 sesiones
1400 de 1651 sesiones
1500 de 1651 sesiones
1600 de 1651 sesiones
(33106, 20)


In [28]:
#Celda 4: puntos del campeonato por temporada

st_rows = []
for _, s in df_seasons.iterrows():
    d = pedir("/results/standings", {"seasonUuid": s["id"], "categoryUuid": cat_motogp[s["id"]]})
    if not isinstance(d, dict):
        continue
    for r in d.get("classification") or []:
        rider = r.get("rider") or {}
        st_rows.append({
            "year": s["year"], "rider_id": rider.get("id"),
            "rider_legacy_id": rider.get("legacy_id"), "piloto": rider.get("full_name"),
            "equipo": (r.get("team") or {}).get("name"),
            "fabricante": (r.get("constructor") or {}).get("name"),
            "pos_campeonato": r.get("position"), "puntos": r.get("points"),
        })

df_stand = pd.DataFrame(st_rows)
print(df_stand.shape)

(322, 8)


In [29]:
# Celda 5: tiempos a segundos y guardar los CSV

import re
import numpy as np

def a_segundos(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x).strip().lstrip("+").replace("'", ":")
    m = re.match(r"^(?:(\d+):)?(\d+(?:\.\d+)?)$", s)
    return np.nan if not m else float(m.group(1) or 0) * 60 + float(m.group(2))

df_clas["mejor_vuelta_s"] = df_clas["mejor_vuelta"].apply(a_segundos)
df_clas["gap_primero_s"] = df_clas["gap_primero"].apply(a_segundos)
df_clas["gap_previo_s"] = df_clas["gap_previo"].apply(a_segundos)

CLEAN = Path("..") / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

df_events.to_csv(CLEAN / "eventos.csv", index=False, encoding="utf-8")
df_ses.to_csv(CLEAN / "sesiones.csv", index=False, encoding="utf-8")
df_clas.to_csv(CLEAN / "clasificaciones.csv", index=False, encoding="utf-8")
df_stand.to_csv(CLEAN / "standings.csv", index=False, encoding="utf-8")

for f in sorted(CLEAN.glob("*.csv")):
    print(f.name, "-", round(f.stat().st_size / 1024), "KB")

clasificaciones.csv - 7905 KB
eventos.csv - 37 KB
sesiones.csv - 244 KB
standings.csv - 30 KB
